# Your Local Evaluator Is Probably Measuring a Different Game

**My first local evaluator correlated with the real ladder at r = +0.014 across 51 board seeds I could
check against actual episodes — not a weak instrument, a disconnected one. Thirteen hand-designed changes
and six training runs were evaluated on it, and every one of those verdicts was noise. Fixing the
measurement, not inventing strategy, is what moved me from 5045/7624 to 1050/9803.**

| | |
|:--|:--|
| **The broken instrument** | vs `starter`: mean bank **$121k**. Same agent, same seeds, real ladder: **$81k**. corr **+0.014** |
| **The fixed instrument** | contested league on real ladder seeds: corr(bank) **+0.458**, corr(margin) **+0.533** |
| **The metric** | the final standing is a **Bradley-Terry** fit on win/loss, so margin is a diagnostic and win rate is the objective |
| **Measured dead ends** | **15**, listed with confidence intervals. Two things worked. |
| **Rank** | 5045/7624 (top 66%) → **1050/9803** (top 10.7%) |
| **Runtime** | standard library + `kaggle_environments`, no internet, ~2 minutes |

**To use it:** *Copy & Edit → Save & Run All*. Nothing here submits anything — this is a method, not an agent.

The agent I currently submit is a fork of a public Apache-2.0 notebook, attributed, with its upstream
notices intact: **the farming is not mine**. What this notebook adds is how to tell whether a change to
an agent actually helped, and a list of the things that turned out not to.

## 1 · What this is

Three weeks of this competition taught me one thing worth publishing, and it is not a strategy.
It is that in a two-player economic sim, **the opponent is part of the environment**, and an evaluator
that forgets this measures a game nobody is playing.

What follows is the harness I ended up with, the check that tells you whether your own harness is
connected to reality, and every change I measured that did not work. The negative results are the
larger half and the more useful one.

What is deliberately **not** here: which public agents are currently strongest, or any ranking of them.
The competition is still running.

## 2 · Your evaluator is probably disconnected

The market price of a good moves with how much of it has been sold. When your opponent sells, your
prices move too. So an opponent who never trades leaves you farming an **uncontested** market, which is
a different economy with different optimal play.

I measured this the expensive way. Running my own submission on the exact board seeds of its own ladder
episodes gave mean bank **$121k** against `starter` and **$81k** on the real ladder — the same agent, the
same boards. A live opponent was costing about **$40k an episode**, a third of the bank, and none of my
local numbers knew about it.

The next cell demonstrates the effect with agents everyone has: it plays `starter` against `random`
(an opponent that barely trades) and `starter` against itself (an opponent that trades exactly as hard
as you do), on identical seeds, and prints the gap.

In [ ]:
# WHAT: play one agent on identical boards against a barely-trading opponent and
# against a fully-trading one, and compare its bank.
# WHY: if the two numbers differ a lot, every result you have ever measured
# against a non-trading opponent is about a different economy.
from kaggle_environments import make

SEEDS = [11, 23, 57, 91]          # fixed so both conditions see identical boards


def bank_of(agent, opponent, seeds):
    '''Mean bank for `agent` in seat 0, over `seeds`. Same boards every time.'''
    out = []
    for s in seeds:
        env = make("kaggriculture",
                   configuration={"episodeSteps": 720, "seed": s}, debug=False)
        env.run([agent, opponent])
        out.append(env.steps[-1][0]["reward"])   # seat 0 final coins
    return sum(out) / len(out)


# `random` places few sensible orders, so the market stays nearly uncontested.
uncontested = bank_of("starter", "random", SEEDS)
# `starter` vs itself: both sides sell into the same book on the same schedule.
contested = bank_of("starter", "starter", SEEDS)

print("uncontested (vs random) mean bank %.0f" % uncontested)
print("contested   (vs self)   mean bank %.0f" % contested)

# IMPORTANT: the built-in `starter` barely farms -- it finishes near the 3000
# starting money -- so it has almost nothing to sell and cannot move prices.
# The gap only appears for an agent that actually trades. Swap your own in.
if uncontested < 2 * 3000:
    print("`starter` banks near the 3000 start: too weak to contest anything.")
    print("substitute your own agent above -- mine showed 121k vs 81k on real boards.")
else:
    print("gap %.0f (%.0f%%)" % (uncontested - contested,
                                 100.0 * (uncontested - contested) / uncontested))

## 3 · Checking a harness against the ladder

A harness you never check is a harness you are trusting on faith. Mine sat at r = +0.014 for a week
without complaining.

The check is possible because Kaggle will tell you exactly what happened in your own episodes. Two
internal endpoints answer with a bearer token from `~/.kaggle/`:

- `competitions.EpisodeService/ListEpisodes` with `{"submissionId": N}` — every episode, each agent's
  final `reward`, and their `initialScore` (the opponent's rating going in).
- `competitions.EpisodeService/GetEpisode` with `{"episodeId": N}` — the same plus the **board seed**.

The seed is the important one. With it you can replay your own ladder boards locally and ask whether
your harness's opinion of board 2059063135 matches what actually happened on board 2059063135. That is
a correlation you can compute, and it is the only honest way to know your instrument works.

The next cell is that fetch. It needs your own credentials and internet, so it is written to skip
cleanly when run in a notebook without them.

In [ ]:
# WHAT: pull your own ladder episodes -- per-agent reward, opponent rating, board seed.
# WHY: these are the ground truth you validate a local harness against. Without
# them you are comparing your simulator to your simulator.
import json, os, urllib.request

API = "https://www.kaggle.com/api/i/competitions.EpisodeService/"


def _token():
    '''Bearer token from ~/.kaggle/access_token, or None if unavailable.'''
    p = os.path.expanduser("~/.kaggle/access_token")
    if not os.path.exists(p):
        return None
    raw = open(p).read().strip()
    try:
        return json.loads(raw).get("access_token") or json.loads(raw).get("token")
    except Exception:
        return raw                      # older tokens are stored bare


def call(method, body, token):
    req = urllib.request.Request(
        API + method, data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json",
                 "User-Agent": "Mozilla/5.0",
                 "Authorization": "Bearer " + token})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)


def my_episodes(submission_id, token):
    '''(opponent_rating, my_bank, their_bank) per completed episode.'''
    out = []
    for e in call("ListEpisodes", {"submissionId": submission_id}, token)["episodes"]:
        if e.get("state") != "COMPLETED":
            continue
        me = [a for a in e["agents"] if a.get("submissionId") == submission_id]
        op = [a for a in e["agents"] if a.get("submissionId") != submission_id]
        if me and op and me[0].get("reward") is not None:
            out.append((op[0].get("initialScore"), me[0]["reward"], op[0]["reward"]))
    return out


tok = _token()
if tok is None:
    print("no ~/.kaggle/access_token here -- run this locally with your own token")
    print("then: my_episodes(<your submission id>, tok)")
else:
    print("token found; call my_episodes(<your submission id>, tok)")

## 4 · Measure win rate, not margin

From the host post on final evaluation (María Cruz, Kaggle staff, 2026-07-31):

> "At the submission deadline, we will continue allowing submissions to run episodes for two weeks."
> "we will be running a single Bradley-Terry Tournament, which will determine the final leaderboard rankings."

Bradley-Terry is fit on **win/loss**. A competitor reported losing an episode 103,148 to 103,147 and
taking a full rating penalty for it — one coin. So a change that adds money to games you already win is
worth nothing, and a change that flips close losses is worth everything.

I had been ranking candidates on mean margin. Switching the headline statistic to win rate **changed
which agent I shipped**. Two real candidates from my own run, anonymised, measured on 384 paired games:

| | margin | win rate |
|:--|:--|:--|
| agent A | **+$27,682** | 91.7% |
| agent B | +$25,715 | **97.7%** |

Margin says A. Win rate says B. Played directly against each other on identical boards, B beats A by
**+6.0 pp [95% CI +2.9, +9.4]** while banking **$1,967 less**: it wins more often, by smaller amounts.

The next cell is the function that computes both, so the disagreement is visible rather than implied.

In [ ]:
# WHAT: given two agents' results on the SAME boards, report both deltas.
# WHY: margin and win rate can point at different agents. Print both, decide on
# the one the competition actually pays for -- here, win rate.
def paired_deltas(a, b):
    '''a, b: {board_key: (my_bank, their_bank)} measured on identical boards.'''
    shared = sorted(set(a) & set(b))
    if not shared:
        return None
    win = lambda r: 1.0 if r[0] > r[1] else 0.0        # noqa: E731
    dwin = sum(win(a[k]) - win(b[k]) for k in shared) / len(shared)
    dmar = sum((a[k][0] - a[k][1]) - (b[k][0] - b[k][1]) for k in shared) / len(shared)
    return {"n": len(shared), "dwin_pp": 100.0 * dwin, "dmargin": dmar}


# A worked example built to the shape my real pair had: B wins more games,
# A banks more per game. 50 boards so a 6 pp difference is representable.
A, B = {}, {}
for i in range(50):
    if i < 43:                       # both win: A wins by more
        A[i], B[i] = (110500, 90000), (108000, 90000)
    elif i < 46:                      # close ones B converts and A does not
        A[i], B[i] = (94000, 95000), (96000, 95000)
    else:                             # both lose
        A[i], B[i] = (80000, 99000), (80000, 99000)
d = paired_deltas(B, A)
print("B vs A on %d identical boards: dwin %+.1f pp, dmargin %+.0f"
      % (d["n"], d["dwin_pp"], d["dmargin"]))
print("margin prefers A; win rate prefers B; Bradley-Terry pays for B")

## 5 · The paired harness

Four properties, each of which I added only after being burned without it:

1. **Opponents contest the market.** Never `starter`, never `random`, never a frozen replay of a game
   that already happened. Use agents that actually trade.
2. **Boards are real ladder seeds**, taken from your own episodes (section 3), not `range(24)`. You
   want the distribution Kaggle deals, not the one you picked.
3. **Every candidate sees identical (seed, opponent, seat) triples, and both seats are played.**
   Pairing cancels board luck; playing both seats cancels seat advantage. This is what takes the
   interval from "±14 points" to something that can resolve a 1% change.
4. **A bootstrap CI on the paired difference**, and a floor below which you call it a no-op regardless.
   Without that floor an exact no-op once reported "BETTER" to me on a $1 delta, because identical
   paired deltas give the bootstrap no spread to find.

The next cell is the whole harness, about 40 lines. It ships no opponents — supply your own.

In [ ]:
# WHAT: paired A/B harness. Same boards, same opponents, both seats, bootstrap CI.
# WHY: an unpaired comparison on 24 boards has a confidence interval so wide it
# will confirm whatever you hoped. Pairing is what buys resolution.
import random
from kaggle_environments import make


def play(agent, opponent, seed, seat):
    '''One episode; returns (agent_bank, opponent_bank). `seat` 0 or 1.'''
    env = make("kaggriculture",
               configuration={"episodeSteps": 720, "seed": seed}, debug=False)
    env.run([agent, opponent] if seat == 0 else [opponent, agent])
    final = env.steps[-1]
    return final[seat]["reward"], final[1 - seat]["reward"]


def evaluate(agent, opponents, seeds):
    '''Play every (seed, opponent, seat). The key identifies the board exactly,
    so two agents' results line up for pairing.'''
    return {"%s|%s|%d" % (op, s, seat): play(agent, op, s, seat)
            for s in seeds for op in opponents for seat in (0, 1)}


def bootstrap(deltas, iters=4000, seed=12345):
    '''95% CI of the mean of `deltas` by resampling with replacement.'''
    rnd, n, means = random.Random(seed), len(deltas), []
    for _ in range(iters):
        means.append(sum(deltas[rnd.randrange(n)] for _ in range(n)) / n)
    means.sort()
    return means[int(0.025 * iters)], means[int(0.975 * iters)]


def compare(cand, base, opponents, seeds):
    '''Win-rate delta of `cand` over `base`, with a CI, on identical boards.'''
    ca, ba = evaluate(cand, opponents, seeds), evaluate(base, opponents, seeds)
    keys = sorted(set(ca) & set(ba))
    win = lambda r: 1.0 if r[0] > r[1] else 0.0        # noqa: E731
    dw = [win(ca[k]) - win(ba[k]) for k in keys]
    lo, hi = bootstrap(dw)
    mean = sum(dw) / len(dw)
    # a change worth under half a point of win rate is noise however tight the CI
    verdict = ("no material difference" if abs(mean) < 0.005 else
               "BETTER" if lo > 0 else "WORSE" if hi < 0 else "not significant")
    return {"n": len(keys), "dwin_pp": 100.0 * mean,
            "ci": (100.0 * lo, 100.0 * hi), "verdict": verdict}


# Demo: `starter` against itself is a true null -- the harness should say so.
r = compare("starter", "starter", ["random"], [11, 23])
print("null check: n=%d dwin %+.1f pp ci [%+.1f, %+.1f] -> %s"
      % (r["n"], r["dwin_pp"], r["ci"][0], r["ci"][1], r["verdict"]))

## 6 · Crash-screen before you believe anything

Most agents wrap their turn in `try/except` and return a PASS action on failure, because a raised
exception forfeits the episode while a no-op costs one turn out of 720. That is correct engineering and
a measurement trap: a agent that throws on **every** turn plays a legal, silent, all-PASS game and banks
the starting money.

That number looks like a result. It is not. This bit me three separate times — once on a variant that
indexed a price array with a shed index, once on a layer that sold the wheat the animals eat, and once
on a path that did not exist. Each produced a plausible-looking bank that I nearly reasoned about.

The tell is simple: a bank at or near the starting money means the agent never farmed. The next cell
checks for it, against a deliberately broken agent and a working one.

In [ ]:
# WHAT: detect agents that silently fail into all-PASS instead of playing.
# WHY: their own try/except hides the exception, and the resulting bank reads as
# a strategy result rather than a crash.
from kaggle_environments import make

START_MONEY = 3000


def broken_agent(obs, config=None):
    '''Deliberately broken: throws every turn, swallows it, returns PASS.'''
    try:
        raise ValueError("pretend this is an index bug you have not noticed")
    except Exception:
        return {"farmer": ["PASS"], "hands": [], "market": []}


def _is_pass(action):
    '''True if this turn did nothing at all.'''
    if not action:
        return True
    return (action.get("farmer") in (None, ["PASS"])
            and not action.get("hands") and not action.get("market"))


def is_alive(agent, seed=11):
    '''Count turns on which the agent actually did something.

    Checking ACTIONS rather than final bank matters: a weak-but-real agent can
    finish near the starting money too, and a bank threshold would call it dead.
    '''
    env = make("kaggriculture",
               configuration={"episodeSteps": 720, "seed": seed}, debug=False)
    env.run([agent, "random"])
    acts = [s[0].get("action") for s in env.steps[1:]]
    active = sum(1 for a in acts if not _is_pass(a))
    return active, env.steps[-1][0]["reward"]


for name, ag in (("broken_agent", broken_agent), ("starter", "starter")):
    active, bank = is_alive(ag)
    print("%-13s acted on %3d of 719 turns, bank %8.0f  %s"
          % (name, active, bank,
             "NOT PLAYING (all-PASS)" if active == 0 else "ok"))

## 7 · Reading a score

The number on the leaderboard is a noisy, lagging view of your agent, and it moves for reasons that have
nothing to do with your code. Measured on my own submissions:

- **A new submission overshoots.** It enters near 600 and climbs fast while its rating is still
  uncertain, so the early number flatters it. One of mine peaked at **1233 around episode 8** and
  settled near **950**; another peaked at **1815** and was at **1539** five days later. The peak is
  never the number to quote. (Another competitor published the same shape from four submissions of
  their own, peaking between games 15 and 21 — worth reading if you want a second sample.)
- **Convergence takes 40–70 episodes**, roughly a day or two. Judging a submission before that, or
  replacing one still climbing, throws the read away. I superseded one of mine 17 minutes after it went
  up, while it was still rising.
- **A frozen agent decays.** One of my submissions fell from 1815 to 1539 in five days — about 55 points
  a day — without a line changing, because the field kept improving and it did not. That stops at the
  deadline, when everyone freezes.

The fix is to stop reading the headline and bucket your episodes by the opponent's rating going in. The
band where you sit near 50% is your actual level. The next cell does that.

In [ ]:
# WHAT: bucket your episodes by opponent rating and report win rate per band.
# WHY: the headline score mixes easy early games with hard later ones. The band
# where you hover near 50% is your real level; everything else is history.
BANDS = [(0, 1500), (1500, 1750), (1750, 2000), (2000, 2250), (2250, 10 ** 9)]


def by_band(episodes):
    '''episodes: (opponent_rating, my_bank, their_bank) -- see section 3.'''
    rows = []
    for lo, hi in BANDS:
        sel = [e for e in episodes if e[0] is not None and lo <= e[0] < hi]
        if not sel:
            continue
        wins = sum(1 for _, mine, theirs in sel if mine > theirs)
        margin = sum(mine - theirs for _, mine, theirs in sel) / len(sel)
        rows.append((lo, hi, len(sel), 100.0 * wins / len(sel), margin))
    return rows


# One of my own converged submissions, 270 episodes, reconstructed to its real
# per-band win rates. It had stopped climbing and was sliding: at the band
# holding most of its games it was winning 42%, not 50%.
def _band(rating, n, win_pct, won, lost):
    '''n episodes at `rating`, `win_pct` of them wins.'''
    w = round(n * win_pct / 100.0)
    return [(rating,) + won] * w + [(rating,) + lost] * (n - w)


example = (_band(1400, 17, 76, (120000, 80000), (70000, 99000))
           + _band(1600, 185, 42, (99000, 95000), (92000, 96000))
           + _band(1850, 67, 28, (97000, 93000), (90000, 95000))
           + _band(2100, 1, 0, (88000, 80000), (88000, 92800)))
for lo, hi, n, win, margin in by_band(example):
    print("band %5d-%-8d n=%-4d win %3.0f%%  margin %+8.0f" % (lo, hi, n, win, margin))
print("50% at your own band means converged; below means the field moved past you")

## 8 · What failed

Fifteen changes, each measured against contesting opponents on real ladder seeds, paired on identical
boards, both seats, with a bootstrap CI. Two worked.

| change | measured | verdict |
|:--|:--|:--|
| **Repair a blocked action instead of losing it** (queue it, clear the blocker, replay it) | **+$4,373 margin, CI [+3345, +5339]**, better on 86% of boards | **worked** |
| **Cutting agent construction cost** (a deep copy on the first callback) | first callback **292–423 ms → 1 ms**, action-for-action identical over 2,157 decisions | **worked — a timeout risk, not a score gain** |
| Idle-unit repairs, when the opponent also repairs | +$112, CI [−995, +1107] | nothing |
| Price-aware sell deferral (hold stock while the price is depressed) | −$2,330 | worse |
| The same, with deferral disabled | +$1 — an exact no-op, which is how I knew the plumbing was right | nothing |
| Handing the endgame to a tuned online controller at day 28 | −$7,087, worse on **0%** of boards | much worse |
| The same at day 25 | −$9,017, worse on 0% of boards | much worse |
| Splitting every large sale across two turns | −$60k to −$80k | catastrophic |
| Capping sale size, spreading sales over three turns | catastrophic | catastrophic |
| Moving every sale one turn earlier | −$80k | catastrophic |
| Selling the shed at the horn | exact no-op — the schedule already emptied it | nothing |
| A cash guard over the next 72 turns of planned buying | **$0** — never fired | nothing |
| A shed-overflow guard | **$0** — never fired | nothing |
| Liquidating stock the plan never sells again | +$63 | nothing |
| Random local mutation of the action schedule | ~$4 per generation | not worth the compute |

Three of those deserve their mechanism stated, because the mechanism is the transferable part:

**The sale schedule sits in a narrow feasible band.** Moving sales *later* fills the shed to capacity,
and a full shed makes the next delivery fail. Moving them *earlier* asks to sell goods that have not
arrived yet, so the order is unfillable and the revenue is simply gone — same quantity requested, $80k
less banked. Both directions fail for different reasons, which is why the schedule looks fragile: it is.

**Guards that never fire are telling you something.** The cash guard and the shed guard measured exactly
$0 because the schedule they were protecting never ran short of cash and never filled the shed. I had
ported them from stronger agents; they were tuned to failure modes my agent did not have.

**A well-regarded public agent measured worse than mine; another measured far better.** Reputation and
headline score did not predict which. Re-measure every candidate yourself.

## 9 · Do's and don'ts

**Do**

- Validate your harness against real episodes before you trust a single verdict from it. Correlation
  with actual outcomes on actual boards, or it does not count.
- Pair everything: identical seeds, identical opponents, both seats.
- Report the statistic the competition pays for. Here that is win rate.
- Crash-screen every variant and every third-party agent before it influences anything.
- Write down what you expect *before* you submit. A prediction you recorded is the only cheap way to
  find out that your instrument is lying.

**Don't**

- Don't measure against `starter`, `random`, `pass`, or a frozen replay and call the result an
  improvement. That is a different economy.
- Don't read a submission before ~40 episodes, and don't replace one that is still climbing.
- Don't infer quality from a headline score or a notebook title. Measure it.
- Don't trust a bank near the starting money. That is a crash wearing a result's clothes.
- Don't let a caching or logging failure kill a measurement run — or, worse, silently truncate it.
  Mine crashed mid-loop on a file lock and printed a partial table that looked complete.

**Still open**, honestly

- Whether the final Bradley-Terry fit uses only post-deadline episodes or the whole history. Asked
  three times on the forum; unanswered when I checked.
- Whether a genuinely adaptive agent beats a good fixed schedule here. I never got one to. The tuned
  online controller I built lost on 100% of boards against a recorded schedule, even given only the
  last 48 turns of 720.

## 10 · Credits and license

The measurement work here is mine. Almost nothing else is.

- The **daily episode replay datasets** published by the hosts are what make any of this checkable, and
  the `EpisodeService` endpoints in section 3 are the same ones the site itself uses.
- The **final-evaluation post** by María Cruz (Kaggle staff, 2026-07-31) is the source for the
  Bradley-Terry statement in section 4. If you have been optimising mean margin, read it.
- Community notebooks measuring **what a replayed action schedule actually recovers**, and the
  published **reconstruction of the market price function**, saved me from two wrong turns each.
- The agent I currently submit is a **fork of a public Apache-2.0 notebook by another competitor**,
  used under that licence with its `NOTICE` and upstream attributions intact, and credited in my
  submission description. The farming logic in it is not mine and I do not claim it.

This notebook is released under **Apache-2.0**. Copy the harness, argue with the numbers, and please
publish it when your measurements disagree with mine — that is more useful to me than agreement.